In [1]:
import h5py
import numpy as np
import os
import sys
import time
import multiprocessing
from functools import partial
from tqdm.auto import tqdm

# Import your tools
sys.path.insert(0, os.path.abspath(".."))
from tools import utils

In [2]:
# ==============================================================================
# CONFIGURATION SECTION
# ==============================================================================

# 1. SIMULATION INFO
SNAP_NUM = "264"          
# CHANGED: Updated to "fiducial" to match the likely name of your xmatch file
SIM_NAME = "fiducial" 
BASE_DATA_PATH = "/home/STORAGE/MN5_resims/fiducial/hydro_output" 

# 2. SELECTION MODE
USE_XMATCH = True # True to use Francisco's matched halos list

# 3. OUTPUT DESTINATION
try:
    PROJ_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

LIBRARY_OUTPUT_DIR = os.path.join(PROJ_ROOT, f"notebooks/mini_sims_library_{SIM_NAME}_100Mpch")
XMATCH_DIR = os.path.join(PROJ_ROOT, "xmatchs")

# 4. BOX GEOMETRY (FIXED L = 100 Mpc/h)
# We confirmed the native unit is cMpc/h. So 100 Mpc/h is exactly 100.0 native.
TARGET_BOX_NATIVE = 100.0

# 5. PARALLELISM
NUM_WORKERS = 20  # Using 20 out of 32 available cores

# ==============================================================================

def smart_read(dataset, mask):
    """Reads dataset with fallback to uint64 if it is uint48 (<u6)."""
    try:
        data = dataset[:]
        return data[mask]
    except TypeError:
        shape = dataset.shape
        buffer = np.empty(shape, dtype=np.uint64)
        dataset.read_direct(buffer)
        return buffer[mask]

def get_attr_with_fallback(f_in, attr_name):
    """Helper to safely get attributes from MN5 headers/parameters."""
    if 'Header' in f_in and attr_name in f_in['Header'].attrs:
        return f_in['Header'].attrs[attr_name]
    if 'Parameters' in f_in and attr_name in f_in['Parameters'].attrs:
        return f_in['Parameters'].attrs[attr_name]
    return None

def process_single_halo(task_index, output_root, target_box_native):
    """
    Worker Function: Extracts a strictly bounded 100 Mpc/h cubic region.
    """
    try:
        ctx = utils.SimulationContext(BASE_DATA_PATH, SNAP_NUM)
        halo_finder = utils.HaloFinder(ctx)
        
        if USE_XMATCH:
            matcher = utils.HaloMatcher(XMATCH_DIR)
            real_halo_id = matcher.get_halo_id(SIM_NAME, task_index)
            halo_lookup_id = real_halo_id
        else:
            real_halo_id = task_index
            halo_lookup_id = task_index

        try:
            halo_props = halo_finder.get_halo_properties(halo_lookup_id)
        except ValueError:
             return f"[SKIP] Halo {real_halo_id} not found/out of bounds."

        # Filter invalid halos
        mass_val = halo_props.get('mass', 0.0)
        if mass_val <= 0:
            return f"[SKIP] Halo {real_halo_id} is invalid (Mass={mass_val:.2e})"

        halo_dir_name = f"halo_{real_halo_id}"
        current_output_dir = os.path.join(output_root, halo_dir_name)
        
        final_file = os.path.join(current_output_dir, f"snapdir_{ctx.snap_num}", f"snapshot_{ctx.snap_num}.0.hdf5")
        if os.path.exists(final_file):
            return f"[SKIP] Halo {real_halo_id} already exists."

        # E. GEOMETRY & UNITS LOGIC
        # Usually 500.0 cMpc/h for MN5
        native_box_size = get_attr_with_fallback(h5py.File(ctx.get_snap_file(0), 'r'), 'BoxSize') 
        if native_box_size is None: native_box_size = 500.0 
        
        halo_center_native = halo_props['cm']
        half_box_native = target_box_native / 2.0
        
        # New center is at exactly L/2 for all axes in the mini-box
        new_center_pos = np.array([half_box_native, half_box_native, half_box_native])

        snap_dir_name = f"snapdir_{ctx.snap_num}"
        group_dir_name = f"groups_{ctx.snap_num}"
        os.makedirs(os.path.join(current_output_dir, snap_dir_name), exist_ok=True)
        os.makedirs(os.path.join(current_output_dir, group_dir_name), exist_ok=True)

        # --- PART 1: SNAPSHOT EXTRACTION ---
        out_snap_file = os.path.join(current_output_dir, snap_dir_name, f"snapshot_{ctx.snap_num}.0.hdf5")
        first_snap = ctx.get_snap_file(0)
        
        with h5py.File(first_snap, 'r') as f_in, h5py.File(out_snap_file, 'w') as f_out:
            # 1.1 Copy and patch Header
            header = f_out.create_group("Header")
            for k, v in f_in['Header'].attrs.items(): header.attrs[k] = v
            header.attrs['BoxSize'] = target_box_native
            header.attrs['NumFilesPerSnapshot'] = 1
            
            # 1.2 Copy and patch Parameters
            if "Parameters" in f_in:
                f_in.copy("Parameters", f_out)
                if "BoxSize" in f_out["Parameters"].attrs:
                    f_out["Parameters"].attrs["BoxSize"] = target_box_native
            else:
                params = f_out.create_group("Parameters")
                for k in ['HubbleParam', 'Omega0', 'OmegaLambda', 'UnitLength_in_cm', 'UnitMass_in_g', 'UnitVelocity_in_cm_per_s']:
                    val = get_attr_with_fallback(f_in, k)
                    if val is not None: params.attrs[k] = val
                params.attrs["BoxSize"] = target_box_native

            n_files_orig = int(f_in['Header'].attrs['NumFilesPerSnapshot'])
            total_parts_by_type = np.zeros(6, dtype=np.int64)

            # 1.3 Read and filter particles
            for pt_idx in range(6):
                pt_name = f"PartType{pt_idx}"
                accumulated_data = {}
                has_data = False
                
                for i in range(n_files_orig):
                    try:
                        f_chunk = h5py.File(ctx.get_snap_file(i), 'r')
                    except: continue
                    
                    if pt_name not in f_chunk: 
                        f_chunk.close(); continue

                    coords = f_chunk[pt_name]['Coordinates'][:]
                    
                    # Compute distance to center, accounting for Periodic Boundary Conditions
                    dx = coords - halo_center_native
                    dx[dx > native_box_size/2.0] -= native_box_size
                    dx[dx < -native_box_size/2.0] += native_box_size
                    
                    # Create strict spatial mask for the fixed 100 Mpc/h box
                    mask = (np.abs(dx[:, 0]) <= half_box_native) & \
                           (np.abs(dx[:, 1]) <= half_box_native) & \
                           (np.abs(dx[:, 2]) <= half_box_native)
                    
                    if np.sum(mask) > 0:
                        has_data = True
                        # Shift coordinates so bottom-left is (0,0,0) and target halo is at L/2
                        new_coords = dx[mask] + new_center_pos
                        
                        for key in f_chunk[pt_name].keys():
                            if key not in accumulated_data: accumulated_data[key] = []
                            if key == 'Coordinates': 
                                accumulated_data[key].append(new_coords)
                            else: 
                                dset = f_chunk[pt_name][key]
                                accumulated_data[key].append(smart_read(dset, mask))
                    f_chunk.close()
                
                if has_data:
                    g_out = f_out.create_group(pt_name)
                    for key, data_list in accumulated_data.items():
                        full_data = np.concatenate(data_list, axis=0)
                        g_out.create_dataset(key, data=full_data, compression="gzip")
                        if key == 'Coordinates': total_parts_by_type[pt_idx] = full_data.shape[0]

            header.attrs['NumPart_ThisFile'] = total_parts_by_type
            header.attrs['NumPart_Total'] = total_parts_by_type
            header.attrs['NumPart_Total_HighWord'] = np.zeros(6, dtype=np.int64)

        # --- PART 2: GROUP FILE CREATION ---
        out_group_file = os.path.join(current_output_dir, group_dir_name, f"fof_subhalo_tab_{ctx.snap_num}.0.hdf5")
        
        with h5py.File(ctx.get_group_file(0), 'r') as g_in, h5py.File(out_group_file, 'w') as g_out:
            header = g_out.create_group("Header")
            for k, v in g_in['Header'].attrs.items(): header.attrs[k] = v
            
            header.attrs['BoxSize'] = target_box_native
            header.attrs['Ngroups_Total'] = 1
            header.attrs['Ngroups_ThisFile'] = 1
            header.attrs['NumFiles'] = 1
            
            if "Parameters" in g_in: g_in.copy("Parameters", g_out)

            grp = g_out.create_group("Group")
            grp.create_dataset("GroupPos", data=np.array([new_center_pos]))
            grp.create_dataset("Group_M_Crit200", data=np.array([halo_props['mass']]))
            grp.create_dataset("Group_R_Crit200", data=np.array([halo_props['r200']]))
            
            # Assuming Time is scale factor a
            scale_factor = get_attr_with_fallback(g_in, 'Time')
            if scale_factor is None: scale_factor = 1.0 
            
            raw_vel_native = halo_props['vel_pec'] / np.sqrt(scale_factor)
            grp.create_dataset("GroupVel", data=np.array([raw_vel_native]))

        return f"[OK] Halo {real_halo_id} Box: {TARGET_BOX_NATIVE} Mpc/h"

    except Exception as e:
        return f"[ERROR] Task {task_index}: {str(e)}"

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
def main():
    print("==================================================")
    print(f"   MINI-SIMULATION GENERATOR (MN5 - Fixed 100 Mpc/h Box)")
    print(f"   Simulation : {SIM_NAME}")
    print(f"   Path       : {BASE_DATA_PATH}")
    print(f"   Snapshot   : {SNAP_NUM} (z=0)")
    print(f"   Output     : {LIBRARY_OUTPUT_DIR}")
    print(f"   Workers    : {NUM_WORKERS}")
    print(f"   Box Size   : {TARGET_BOX_NATIVE} cMpc/h")
    print("==================================================\n")
    
    os.makedirs(LIBRARY_OUTPUT_DIR, exist_ok=True)
    
    # 1. Determine Task List
    if USE_XMATCH:
        try:
            matcher = utils.HaloMatcher(XMATCH_DIR)
            cat_len = len(matcher.match_catalogs.get(SIM_NAME, []))
            if cat_len == 0: 
                 matcher.get_halo_id(SIM_NAME, 0)
                 cat_len = len(matcher.match_catalogs[SIM_NAME])
            ALL_TASKS = list(range(cat_len))
            print(f"--> Found {len(ALL_TASKS)} halos in Xmatch catalog.")
        except Exception as e:
            print(f"[CRITICAL] Xmatch setup failed: {e}")
            print("Make sure your SIM_NAME variable matches the name in xmatch_<SIM_NAME>.npy")
            sys.exit(1)
    else:
        try:
            temp_ctx = utils.SimulationContext(BASE_DATA_PATH, SNAP_NUM)
            total_halos = temp_ctx.get_total_groups()
            ALL_TASKS = list(range(total_halos))
            print(f"--> Found {len(ALL_TASKS)} total halos in simulation header.")
        except Exception as e:
            print(f"[CRITICAL] Failed to read simulation header: {e}")
            sys.exit(1)

    # 2. Run Parallel execution
    start_time = time.time()
    
    worker_func = partial(process_single_halo, output_root=LIBRARY_OUTPUT_DIR, target_box_native=TARGET_BOX_NATIVE)
    
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        # Using tqdm.auto for smooth progress tracking
        results = list(tqdm(pool.imap_unordered(worker_func, ALL_TASKS), total=len(ALL_TASKS), desc="Extracting Halos", unit="halo"))
    
    # 3. Report
    print("\n=== FINAL REPORT ===")
    errors = [r for r in results if "[ERROR]" in r]
    success = [r for r in results if "[OK]" in r]
    skips = [r for r in results if "[SKIP]" in r]
    
    print(f"Success : {len(success)}")
    print(f"Skipped : {len(skips)} (Invalid Halos or Already Done)")
    print(f"Errors  : {len(errors)}")
    
    if errors:
        print("\n--- ERROR SAMPLES ---")
        for err in errors[:5]: print(err)

    elapsed_time = (time.time() - start_time) / 60
    print(f"\nTotal Time: {elapsed_time:.2f} min")


In [3]:
if __name__ == "__main__":
    main()

2026-03-13 17:08:33,226 - INFO - Loading match catalog for fiducial...


   MINI-SIMULATION GENERATOR (MN5 - Fixed 100 Mpc/h Box)
   Simulation : fiducial
   Path       : /home/STORAGE/MN5_resims/fiducial/hydro_output
   Snapshot   : 264 (z=0)
   Output     : /home/vroberto/FakeSpectra_DLAs/notebooks/mini_sims_library_fiducial_100Mpch
   Workers    : 20
   Box Size   : 100.0 cMpc/h

--> Found 452 halos in Xmatch catalog.


2026-03-13 17:08:33,267 - INFO - Loading match catalog for fiducial...


Extracting Halos:   0%|          | 0/452 [00:00<?, ?halo/s]

2026-03-13 17:08:33,267 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,270 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,270 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,270 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,270 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,271 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,271 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,273 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,272 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,273 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,273 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,273 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,274 - INFO - Loading match catalog for fiducial...
2026-03-13 17:08:33,274 - INFO - Loading match catalog for fiducial...
2026-0


=== FINAL REPORT ===
Success : 452
Skipped : 0 (Invalid Halos or Already Done)
Errors  : 0

Total Time: 33.39 min
